In [ ]:
import requests
import smtplib
import schedule
import time
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from twilio.rest import Client

# Configuração dos destinatários
EMAIL_REMETENTE = "seu_email@gmail.com"
SENHA_EMAIL = "sua_senha"
EMAIL_DESTINATARIOS = ["destinatario1@gmail.com", "destinatario2@gmail.com"]
SMTP_SERVER = "smtp.gmail.com"
SMTP_PORT = 587

ACCOUNT_SID = "seu_account_sid"
AUTH_TOKEN = "seu_auth_token"
NUMERO_REMETENTE_WSP = "whatsapp:+seu_numero_twilio"
NUMEROS_DESTINATARIOS_WSP = ["whatsapp:+numero1", "whatsapp:+numero2"]

# Dataframe de títulos e URLs
dados = [
    {"id_titulos": 1, "symbol": "DXY", "url": "https://www.forecasts.org/dollar-forecast.htm"},
    {"id_titulos": 2, "symbol": "US10Y", "url": "https://www.forecasts.org/10yrT.htm"},
    {"id_titulos": 3, "symbol": "USIR", "url": "https://www.forecasts.org/inflation.htm"},
    {"id_titulos": 4, "symbol": "SPX", "url": "https://www.forecasts.org/stpoor.htm"},
    {"id_titulos": 5, "symbol": "EUR/USD", "url": "https://www.forecasts.org/euro.htm"},
]

def consultar_paginas():
    mensagens = []
    for dado in dados:
        try:
            resposta = requests.get(dado["url"])
            if resposta.status_code == 200:
                mensagem = f"O forecast para {dado['symbol']} foi atualizado com sucesso!"
            else:
                mensagem = f"Erro ao acessar {dado['symbol']}: código {resposta.status_code}"
        except Exception as e:
            mensagem = f"Erro ao acessar {dado['symbol']}: {e}"

        mensagens.append(mensagem)
    
    return mensagens

def enviar_email(mensagens):
    try:
        msg = MIMEMultipart()
        msg["From"] = EMAIL_REMETENTE
        msg["To"] = ", ".join(EMAIL_DESTINATARIOS)
        msg["Subject"] = "Confirmação de Forecasts"

        texto_email = "\n".join(mensagens)
        msg.attach(MIMEText(texto_email, "plain"))

        server = smtplib.SMTP(SMTP_SERVER, SMTP_PORT)
        server.starttls()
        server.login(EMAIL_REMETENTE, SENHA_EMAIL)
        server.sendmail(EMAIL_REMETENTE, EMAIL_DESTINATARIOS, msg.as_string())
        server.quit()

        print("E-mail enviado com sucesso!")
    except Exception as e:
        print(f"Erro ao enviar e-mail: {e}")

def enviar_whatsapp(mensagens):
    try:
        client = Client(ACCOUNT_SID, AUTH_TOKEN)
        for numero in NUMEROS_DESTINATARIOS_WSP:
            for mensagem in mensagens:
                client.messages.create(
                    from_=NUMERO_REMETENTE_WSP,
                    body=mensagem,
                    to=numero
                )
        print("Mensagens WhatsApp enviadas com sucesso!")
    except Exception as e:
        print(f"Erro ao enviar WhatsApp: {e}")

def executar_processo():
    mensagens = consultar_paginas()
    enviar_email(mensagens)
    enviar_whatsapp(mensagens)

# Agendar execução diária às 09:00
schedule.every().day.at("09:00").do(executar_processo)

while True:
    schedule.run_pending()
    time.sleep(60)